# P3: Compilación y Síntesis del Modelo Student con hls4ml y Vitis HLS

<table style="width:100%; background-color:#2E1065;">
<tr>
<td width="70%">

**PyCon Colombia 2026 🇨🇴 - Workshop | hls4ml: From Python Models to Hardware Acceleration**  
*Versión para Google Colab*

</td>

<td width="30%" align="center" valign="middle">

<img src="../imgs/pycon_logo.png" width="100">
&nbsp;&nbsp;&nbsp;
<img src="../imgs/gicm_logo.png" width="100">

</td>
</tr>
</table>

En este *notebook* se realiza la conversión, configuración y compilación del modelo **Student** optimizado (previamente podado y cuantizado) para transformarlo en un diseño de hardware sintetizable en C++ mediante **Vitis HLS**.

* **Traducción a High Level Synthesis (HLS):** Uso de `hls4ml` para el mapeo de las capas Keras/QKeras del modelo a descripciones en C++ aptas para ser implementadas en hardware.

* **Configuración de Precisión:** Ajuste fino de parámetros mediante tipos de punto fijo (`ap_fixed`) y estrategias de latencia.

* **Síntesis:** Ejecución de scripts Tcl para la síntesis en Vitis HLS.

---

### Referencias

Flujo de trabajo basado en

- R. S. Molina, I. R. Morales, M. L. Crespo, V. G. Costa, S. Carrato y G. Ramponi, "An End-to-End Workflow to Efficiently Compress and Deploy DNN Classifiers on SoC/FPGA", en IEEE Embedded Systems Letters, vol. 16, no. 3, pp. 255-258, septiembre 2024, doi: 10.1109/LES.2023.3343030.

Código adaptado del repositorio oficial de

- "An End-to-End Workflow to Efficiently Compress and Deploy DNN Classifiers on SoC/FPGA".

Documentación Oficial de hls4ml: https://fastmachinelearning.org/hls4ml/

<div align="center">
  <img src="../imgs/hls4ml_integration.png" alt="Arquitectura de integración con hls4ml" width="80%">
  <br>
  <em>Figura 1: Flujo para compilación y síntesis mediante hls4ml.</em>
</div>

In [ ]:
# ============================================================
# Setup P3: Preparación del entorno
# ============================================================

import os
# 1. Instalar las dependencias necesarias
!pip install -q tensorflow==2.12.0 keras==2.12.0 qkeras==0.9.0 tensorflow-model-optimization==0.8.0 hls4ml==1.1.0


print("Entorno preparado correctamente.")
print("Directorio actual del kernel:")
print(os.getcwd())

In [ ]:
# 2. Forzamos el reinicio del entorno de ejecución
os.kill(os.getpid(), 9)

In [ ]:
#Verificamos las versiones de Tensorflow y Keras
import tensorflow as tf
import keras

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)

In [ ]:
import os
import platform
import subprocess

import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import *
from tensorflow.keras.layers import *
from qkeras import *
from qkeras import QActivation
from qkeras import QDense, QConv2DBatchnorm

import hls4ml

print("Preparando el entorno...")
# 3. Clonar el repositorio
if not os.path.exists("hls4ml_workshop"):
    !git clone -q https://github.com/GICM-UdeA/hls4ml_workshop.git

# 4. Acceder al directorio del proyecto
os.chdir("hls4ml_workshop")

## 1.) Configuración de Rutas para Vitis HLS

El directorio de Vitis HLS debe especificarse para poder usar `hls4ml` junto con la herramienta de High-Level-Synthesis.

Este bloque de código se encarga de configurar las variables de entorno del sistema operativo (`PATH` y las rutas de Xilinx) para que Python pueda encontrar los ejecutables de **Vitis HLS** y **Xilinx Vivado**. 

> **Nota:**  
> Este bloque se encuentra comentado ya que requiere que la suite completa de Xilinx esté instalada localmente en la computadora. si ejecutas sin Vitis HLS instalado, puedes mantenerlo comentado de forma segura, ya que no afectará la traducción inicial del modelo a C++.

In [ ]:
# ============================================================
# Ruta hacia Vitis HLS (Comentado por falta de instalación local)
# ============================================================
# system = platform.system()
# 
# if system == "Windows":
#     os.environ['PATH'] = '/mnt/d/Xilinx/Vitis_HLS/2022.2/bin:' + os.environ['PATH']
#     os.environ['PATH'] = '/mnt/d/Xilinx/Vitis/2022.2/bin:' + os.environ['PATH']
#     os.environ['PATH'] = '/mnt/d/Xilinx/Vivado/2022.2/bin:' + os.environ['PATH']
# 
# elif system == "Linux":
#     cmd = """
#     source /opt/Xilinx/Vitis_HLS/2022.2/settings64.sh
#     source /opt/Xilinx/Vitis/2022.2/settings64.sh
#     source /opt/Xilinx/Vivado/2022.2/settings64.sh
#     env
#     """
#     proc = subprocess.Popen(["bash", "-c", cmd], stdout=subprocess.PIPE, text=True)
#     for line in proc.stdout:
#         key, _, value = line.partition("=")
#         os.environ[key] = value.strip()
# 
# print("Bloque de entorno de Xilinx omitido (Herramienta no instalada).")

## 2.) Cargar el modelo

Se carga el modelo estudiante previamente entrenado `studentModel_ECG_PTB.h5` para hacer la conversion

Como utilizamos capas personalizadas de QKeras, empleamos un diccionario de objetos compatibles (`_add_supported_quantized_objects`) para asegurar que Keras interprete correctamente las capas cuantizadas al cargar el archivo `.h5`.

In [13]:
# ============================================================
# Carga del modelo Keras personalizado (QKeras)
# ============================================================
from qkeras.utils import _add_supported_quantized_objects

co = {}
_add_supported_quantized_objects(co)

# Cargamos el modelo estudiante específico para el dataset de ECG
model = load_model('./models/studentModel_ECG_PTB.h5', custom_objects=co)

# Mostramos el resumen de la arquitectura y sus parámetros
model.summary()

OSError: No file or directory found at ./models/studentModel_ECG_PTB.h5

## 3.) Integración con `hls4ml`

`hls4ml` es un paquete de Python desarrollado por el CERN para convertir modelos de machine learning (**ML**) en proyectos de HLS (**High-Level Synthesis**), lo que permite desplegar inferencias basadas en ML en hardware como **FPGAs**. Más detalles se pueden encontrar en la documentación oficial de `hls4ml`.

El usuario puede controlar varias opciones relacionadas con el modelo, incluyendo:

* **Precisión**: Permite definir la precisión de los cálculos en el modelo, por ejemplo, usando representación de punto fijo (*fixed-point*) o punto flotante (*floating-point*).
* **Flujo de datos / Reutilización de recursos**: Controla el nivel de paralelismo o *streaming* en la implementación del modelo, permitiendo distintos grados de *pipelining*.

Para implementar un modelo en FPGA, se debe crear una configuración HLS usando la función:

`hls4ml.utils.config_from_keras_model(kerasModel, granularity)`

Donde los parámetros se definen de la siguiente manera:

* **`kerasModel`**: Es el modelo preentrenado que se quiere implementar en la FPGA.
* **`granularity`**: Determina el nivel de configuración, y puede tomar dos valores:
  * `'model'`: La misma configuración se aplica a todo el modelo (por ejemplo, todas las capas usan precisión de 16 bits en punto fijo).
  * `'name'`: Se pueden aplicar configuraciones específicas por capa (por ejemplo, la capa de entrada en 8 bits punto fijo, mientras que la segunda capa en 16 bits punto fijo).

In [ ]:
# ============================================================
# Extracción de la configuración HLS del modelo
# ============================================================
hls_config = hls4ml.utils.config_from_keras_model(model, granularity='name')
print(hls_config)


Interpreting Sequential
Topology:
Layer name: inputLayer, layer type: InputLayer, input shapes: [[None, 187]], output shape: [None, 187]
Layer name: fc1, layer type: QDense, input shapes: [[None, 187]], output shape: [None, 6]
Layer name: relu0, layer type: Activation, input shapes: [[None, 6]], output shape: [None, 6]
Layer name: fc2, layer type: QDense, input shapes: [[None, 6]], output shape: [None, 4]
Layer name: relu1, layer type: Activation, input shapes: [[None, 4]], output shape: [None, 4]
Layer name: fc3, layer type: QDense, input shapes: [[None, 4]], output shape: [None, 2]
Layer name: relu2, layer type: Activation, input shapes: [[None, 2]], output shape: [None, 2]
Layer name: fc4, layer type: QDense, input shapes: [[None, 2]], output shape: [None, 4]
Layer name: relu3, layer type: Activation, input shapes: [[None, 4]], output shape: [None, 4]
Layer name: fc5, layer type: QDense, input shapes: [[None, 4]], output shape: [None, 3]
Layer name: relu4, layer type: Activation, in

Para inspeccionar con mayor claridad la estructura del diccionario de configuración (incluyendo estrategias, factores de reutilización y precisión de cada capa), utilizamos la función auxiliar `print_dict` provista en los scripts de apoyo del flujo.

In [ ]:
# ============================================================
# Visualización legible del diccionario HLS
# ============================================================
from src import plotting

print("-----------------------------------")
plotting.print_dict(hls_config)
print("-----------------------------------")

-----------------------------------
Model
  Precision
    default:         fixed<16,6>
  ReuseFactor:       1
  Strategy:          Latency
  BramFactor:        1000000000
  TraceOutput:       False
LayerName
  inputLayer
    Trace:           False
    Precision
      result:        auto
  fc1
    Trace:           False
    Precision
      result:        auto
      weight:        fixed<8,5,TRN,WRAP,0>
      bias:          fixed<8,5,TRN,WRAP,0>
  fc1_linear
    Trace:           False
    Precision
      result:        auto
  relu0
    Trace:           False
    Precision
      result:        fixed<8,5,RND_CONV,SAT,0>
  fc2
    Trace:           False
    Precision
      result:        auto
      weight:        fixed<8,5,TRN,WRAP,0>
      bias:          fixed<8,5,TRN,WRAP,0>
  fc2_linear
    Trace:           False
    Precision
      result:        auto
  relu1
    Trace:           False
    Precision
      result:        fixed<8,5,RND_CONV,SAT,0>
  fc3
    Trace:           False
    Preci

### 3.1.) Personalización de Parámetros de Hardware (Latencia y Precisión)

En esta celda configuramos los hiperparámetros de síntesis para cada una de las capas de la red:

* **Estrategia de Latencia (`Strategy = 'Latency'`):** Configuramos el hardware para priorizar la velocidad de procesamiento (minimizar los ciclos de reloj), desplegando los operadores aritméticos de forma completamente paralela (*unrolling*).
* **Factor de Reutilización (`ReuseFactor = 1`):** Indica que no se reutilizan recursos aritméticos (multiplicadores/sumadores), maximizando el paralelismo.
* **Precisión por Defecto (`ap_fixed<8,4>`):** Asignamos un formato de punto fijo de 8 bits totales con 4 bits enteros para los pesos y activaciones de la red.
* **Excepciones de Capa:** Ajustamos la capa de salida (`outputActivation`) a una estrategia más estable (`Stable`) y definimos una mayor resolución (`ap_fixed<16,6>`) tanto para la entrada (`inputLayer`) como para el modelo global, garantizando estabilidad numérica en los límites de la red.

In [7]:
for Layer in hls_config['LayerName'].keys():
    hls_config['LayerName'][Layer]['Strategy'] = 'Latency'
    hls_config['LayerName'][Layer]['ReuseFactor'] = 1
    hls_config['LayerName'][Layer]['Precision'] = 'ap_fixed<8,4>'

hls_config['LayerName']['outputActivation']['Strategy'] = 'Stable'
hls_config['Model']['Precision'] = 'ap_fixed<16,6>'

hls_config['LayerName']['inputLayer']['Precision'] = 'ap_fixed<16,6>'

###  hls4ml con Vitis HLS como backend

---

Se debe correr la celda, saldra un error, pero al final es necesario ejecutar en la terminal lo siguiente:


``` bash
export PATH="/d/Xilinx/Vitis_HLS/2022.2/bin/:$PATH"
vitis_hls -version
cd hlsPrj\
vitis_hls -f build_prj.tcl
```

En esta sección configuramos los parámetros de conversión utilizando `hls4ml`, especificando el backend de Vitis HLS, el directorio de salida y la tarjeta FPGA destino (`xc7z020clg484-1`, correspondiente a dispositivos como la PYNQ-Z1 o ZedBoard).

A continuación, ejecutamos la traducción del modelo de Keras a C++ y empleamos el método `.write()` para volcar todos los archivos fuente de hardware directamente en disco, omitiendo los comandos de compilación local que requieren la instalación completa de la suite Vitis HLS (secciones comentadas) ya que no lo tenemos instalado.

In [ ]:
# ============================================================
# Conversión a HLS y Generación de Archivos C++
# ============================================================
cfg = hls4ml.converters.create_config(backend='Vitis')

cfg['HLSConfig']  = hls_config
cfg['KerasModel'] = model
cfg['OutputDir']  = './hlsPrj/'
cfg['Part'] = 'xc7z020clg484-1'  
  
# Esto traduce el modelo de Keras a C++ de forma exitosa sin requerir Vitis instalado
hls_model = hls4ml.converters.keras_to_hls(cfg)

system = platform.system()

# Comentamos la compilación directa por Linux y la ejecución por subprocesos de Windows
# if system == "Linux":
#     hls_model.compile()
# elif system == "Windows":
#     hls_model.write()
#     subprocess.run([r"D:\\Xilinx\\Vitis_HLS\\2022.2\\bin\\vitis_hls.bat", "-f", "hlsPrj/build_prj.tcl"], check=True)
#     subprocess.run(["mv", "./myproject_prj", "./hlsPrj/"], check=True)

# En su lugar, utilizamos únicamente .write() para generar y visualizar el código C++ en disco:
hls_model.write()
print("Modelo convertido a C++ y archivos de hardware generados en './hlsPrj/' con éxito")

Interpreting Sequential
Topology:
Layer name: inputLayer, layer type: InputLayer, input shapes: [[None, 187]], output shape: [None, 187]
Layer name: fc1, layer type: QDense, input shapes: [[None, 187]], output shape: [None, 6]
Layer name: relu0, layer type: Activation, input shapes: [[None, 6]], output shape: [None, 6]
Layer name: fc2, layer type: QDense, input shapes: [[None, 6]], output shape: [None, 4]
Layer name: relu1, layer type: Activation, input shapes: [[None, 4]], output shape: [None, 4]
Layer name: fc3, layer type: QDense, input shapes: [[None, 4]], output shape: [None, 2]
Layer name: relu2, layer type: Activation, input shapes: [[None, 2]], output shape: [None, 2]
Layer name: fc4, layer type: QDense, input shapes: [[None, 2]], output shape: [None, 4]
Layer name: relu3, layer type: Activation, input shapes: [[None, 4]], output shape: [None, 4]
Layer name: fc5, layer type: QDense, input shapes: [[None, 4]], output shape: [None, 3]
Layer name: relu4, layer type: Activation, in

Done


FileNotFoundError: [WinError 2] El sistema no puede encontrar el archivo especificado

## 4.) Ejecución de High Level Synthesis (Omitido)

El método `hls_model.build()` invoca al compilador y sintetizador de Vitis HLS para transformar el código C++ en circuitos digitales, generando métricas detalladas de uso de recursos lógicos (LUTs, FFs, DSPs) y latencia estimada.

> **Nota:**  
> Este bloque se encuentra comentado ya que requiere la instalación local de Vitis HLS. En un entorno con la herramienta instalada, este comando iniciaría el proceso de síntesis en hardware.

In [ ]:
# ============================================================
# Síntesis del Proyecto (Comentado: Requiere Vitis HLS)
# ============================================================
# hls_model.build(csim=False, export=False)
print("La síntesis con `hls_model.build()` está omitida porque Vitis HLS no está instalado en este equipo.")

Project myproject_prj does not exist. Rerun "hls4ml build -p ./hlsPrj/".


### 4.1.) Reemplazar archivos en HLS project

In [ ]:
# subprocess.run(
#     ["cp", "./vitis_src/myproject_test.cpp", "./hlsPrj/"],
#     check=True
# )

# subprocess.run(
#     ["cp", "./vitis_src/myproject.cpp", "./vitis_src/myproject.h", "./hlsPrj/firmware"],
#     check=True
# )

## 5.) Implementacion en Xilinx Vivado


<div align="center">
  <img src="../imgs/VivadoImpl.png" alt="Implementación en Vivado" width="85%">
  <br>
  <em>Figura 2: Diseño de Hardware implementado para la red neuronal a desplegarse en la FPGA.</em>
</div>

---

**Autores**: Natalia Echeverri, Jerónimo López, Fabian Castaño

### **Agradecimientos Especiales**

> **Este taller toma como punto de partida el excelente trabajo de Fabián Andrés Castaño.**
> Gran parte del flujo de trabajo de Machine Learning a Hardware (`hls4ml`) adaptado para la PyCon Colombia está basado en su repositorio original.
>
> * **Repositorio base:** [fabioc9675/ML4HLS_Tutorial_base (GitHub)](https://github.com/fabioc9675/ML4HLS_Tutorial_base)
